# Enveda CASMI 2026 — final inference

Attach the competition data, versioned source-code dataset, October 2026 COCONUT CSV-lite, private 15-feature ranker-training dataset, and offline pinned RDKit wheel dataset as Kaggle Inputs. Keep internet disabled. This notebook writes `submission.csv` to `/kaggle/working`.


In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/casmi26')
WORK.mkdir(parents=True, exist_ok=True)

def one(paths, label):
    paths = sorted(set(paths))
    if len(paths) != 1:
        raise FileNotFoundError(f'Expected one {label}; found {len(paths)}: {paths[:8]}')
    return paths[0]

TRAIN = one(INPUT.rglob('train.parquet'), 'competition train.parquet')
TEST = one(INPUT.rglob('test.parquet'), 'competition test.parquet')
source_files = list(INPUT.rglob('retrieve.py'))
if not source_files:
    source_archives = [p for p in INPUT.rglob('CASMI26_Source_Code_*.zip')]
    source_archive = one(source_archives, 'CASMI26 source-code ZIP')
    source_extract = WORK / 'source_code'
    source_extract.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source_archive) as archive:
        archive.extractall(source_extract)
    source_files = list(source_extract.rglob('retrieve.py'))
source_roots = sorted({p.parent for p in source_files if (p.parent / 'prepare_candidates.py').exists()})
required_source_files = ('retrieve_analogs.py', 'merge_candidate_features.py', 'fit_reranker.py')
compatible_roots = [root for root in source_roots if all((root / name).is_file() for name in required_source_files)]
# Prefer versioned snapshots. Older attached source Datasets may contain a
# source tree alongside a newer ZIP, and should not create false ambiguity.
versioned_roots = [root for root in compatible_roots if (root / 'SOURCE_COMMIT.txt').is_file()]
if versioned_roots:
    compatible_roots = versioned_roots
if len(compatible_roots) != 1:
    details = []
    for root in source_roots:
        commit = (root / 'SOURCE_COMMIT.txt').read_text().strip() if (root / 'SOURCE_COMMIT.txt').is_file() else 'unversioned'
        missing = [name for name in required_source_files if not (root / name).is_file()]
        details.append(f'{root} [commit={commit}; missing={missing}]')
    raise FileNotFoundError(
        'Need exactly one current multichannel source snapshot containing '
        f'{required_source_files}. Compatible snapshots: {compatible_roots}. '
        f'All discovered source roots: {details}. Attach the updated CASMI26 source-code Dataset version.'
    )
SOURCE_ROOT = compatible_roots[0]
coconut_inputs = [p for p in INPUT.rglob('*') if p.is_file() and p.suffix.lower() in {'.zip', '.csv'} and ('coconut' in p.name.lower() or any('coconut' in part.lower() for part in p.parts))]
if not coconut_inputs:
    visible_csv_zip = [str(p) for p in INPUT.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.zip'}][:30]
    raise FileNotFoundError(f'COCONUT CSV/ZIP not found under /kaggle/input. CSV/ZIP files found: {visible_csv_zip}')
COCONUT_ZIP = one(coconut_inputs, 'COCONUT CSV or ZIP')
feature_files = list(INPUT.rglob('ranker_training_features.parquet'))
if not feature_files:
    feature_archives = [p for p in INPUT.rglob('CASMI26_Ranker_Training_Features.zip')]
    feature_archive = one(feature_archives, 'CASMI26 ranker-features ZIP')
    feature_extract = WORK / 'ranker_training_input'
    feature_extract.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(feature_archive) as archive:
        archive.extractall(feature_extract)
    feature_files = list(feature_extract.rglob('ranker_training_features.parquet'))
RANKER_FEATURES = one(feature_files, 'ranker_training_features.parquet')
sys.path.insert(0, str(SOURCE_ROOT))
print('Train:', TRAIN)
print('Test:', TEST)
print('Source:', SOURCE_ROOT)
print('COCONUT:', COCONUT_ZIP)
print('Ranker features:', RANKER_FEATURES)
commit_file = SOURCE_ROOT / 'SOURCE_COMMIT.txt'
print('Source commit:', commit_file.read_text().strip() if commit_file.exists() else 'missing')

In [ ]:
import importlib.util
if importlib.util.find_spec('rdkit') is not None:
    import rdkit
    parts = tuple(int(part) for part in rdkit.__version__.split('.')[:3])
else:
    parts = None
if parts != (2026, 3, 3):
    wheels = [p for p in INPUT.rglob('*.whl') if 'rdkit' in p.name.lower() and ('2026.03.3' in p.name or '2026.3.3' in p.name)]
    if not wheels:
        raise RuntimeError(f'Competition-pinned RDKit 2026.03.3 required; found {parts}. Attach the offline RDKit wheel Kaggle Dataset as an Input.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--upgrade', str(wheels[0])], check=True)
    import rdkit
    parts = tuple(int(part) for part in rdkit.__version__.split('.')[:3])
if parts != (2026, 3, 3):
    raise RuntimeError(f'RDKit upgrade failed; expected 2026.03.3, found {rdkit.__version__}')
print('RDKit:', rdkit.__version__)

## Fit the CPU reranker

The attached feature table is derived from the training set with molecule-grouped validation. The final model is fit on all eligible training groups. This step uses CPU; the GPU is not needed for the current model.

In [ ]:
MODEL = WORK / 'final_reranker.joblib'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'fit_reranker.py'),
    '--features', str(RANKER_FEATURES), '--model-output', str(MODEL),
], check=True)

## Generate candidates and score three evidence channels

Build mass-compatible COCONUT candidates, score observed fragments, transfer evidence from mass-shifted training-library analogs, merge the candidate-level features, and apply the molecule-grouped CPU ranker.


In [ ]:
CANDIDATES = WORK / 'coconut_candidates.parquet'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'prepare_candidates.py'),
    '--input', str(COCONUT_ZIP), '--output', str(CANDIDATES),
    '--query-parquet', str(TEST), '--window-ppm', '50',
    '--source-version', '2026-10', '--workers', '4',
], check=True)

DATABASE_FRAGMENT_FEATURES = WORK / 'database_fragment_features.parquet'
DATABASE_FRAGMENT_PREDICTIONS = WORK / 'database_fragment_predictions.csv'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'retrieve_fragments.py'),
    '--candidates', str(CANDIDATES), '--queries', str(TEST),
    '--window-ppm', '10', '--features-output', str(DATABASE_FRAGMENT_FEATURES),
    '--output', str(DATABASE_FRAGMENT_PREDICTIONS),
], check=True)

DATABASE_ANALOG_FEATURES = WORK / 'database_analog_features.parquet'
DATABASE_ANALOG_PREDICTIONS = WORK / 'database_analog_predictions.csv'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'retrieve_analogs.py'),
    '--train', str(TRAIN), '--queries', str(TEST), '--candidates', str(CANDIDATES),
    '--output-features', str(DATABASE_ANALOG_FEATURES),
    '--output-predictions', str(DATABASE_ANALOG_PREDICTIONS),
    '--batch-size', '4096', '--top-analogs', '128',
], check=True)

DATABASE_FEATURES = WORK / 'database_multichannel_features.parquet'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'merge_candidate_features.py'),
    '--base', str(DATABASE_FRAGMENT_FEATURES),
    '--analog', str(DATABASE_ANALOG_FEATURES),
    '--output', str(DATABASE_FEATURES),
], check=True)

DATABASE_PREDICTIONS = WORK / 'database_predictions.csv'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'predict_reranker.py'),
    '--model', str(MODEL), '--features', str(DATABASE_FEATURES),
    '--output', str(DATABASE_PREDICTIONS),
], check=True)


## Fuse spectral retrieval with the structure-database route

When the competition-key-deduplicated spectral top-two margin is below 0.03, database candidates lead; otherwise spectral candidates lead. Because each route can emit 25 candidates, this is a route switch rather than a score average. The threshold is a validation-informed starting heuristic, not a guarantee; compare the public score and independent holdouts before further tuning.


In [ ]:
SPECTRAL_PREDICTIONS = WORK / 'spectral_predictions.csv'
SPECTRAL_DIAGNOSTICS = WORK / 'spectral_diagnostics.csv'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'retrieve.py'),
    '--train', str(TRAIN), '--test', str(TEST),
    '--output', str(SPECTRAL_PREDICTIONS), '--diagnostics-output', str(SPECTRAL_DIAGNOSTICS),
    '--top-k', '25', '--batch-size', '4096',
], check=True)

BLENDED = WORK / 'blended_predictions.csv'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'blend_predictions.py'),
    '--primary', str(SPECTRAL_PREDICTIONS),
    '--fallback', str(DATABASE_PREDICTIONS), '--test', str(TEST),
    '--diagnostics', str(SPECTRAL_DIAGNOSTICS),
    '--spectral-margin-threshold', '0.03',
    '--output', str(BLENDED),
], check=True)

SUBMISSION = Path('/kaggle/working/submission.csv')
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'write_submission.py'),
    '--predictions', str(BLENDED), '--test', str(TEST),
    '--output', str(SUBMISSION),
], check=True)
print('Submission ready:', SUBMISSION, 'bytes:', SUBMISSION.stat().st_size)
